In [581]:
import numpy as np
from pathlib import Path
import pandas as pd
from itertools import groupby
import pandas as pd
import matplotlib.pyplot as plt
import re
from scipy.ndimage import gaussian_filter1d
from scipy.io import savemat
import herbs
from brainrender import Scene
from brainrender.actors import Points

import json
import pickle

In [582]:
def group_by_mouse_day(path):

    catGT_path = next(part for part in str(path).split("\\") if part.startswith("catgt_"))
    str_groups = re.search(r"catgt_(\d+)_m(\d+)_obs(\d+)", catGT_path)

    mouse_id = str_groups.group(2)
    obs = str_groups.group(3)

    return mouse_id, obs

In [583]:
dt_folder = Path(r"J:\project_trainingAggression\Data")

exclude = ["m1034813", "m1034811", "m1013587"]
 
neuralDataPaths = [f for f in dt_folder.rglob("autoForNeuroConcat") if not any(x in str(f) for x in exclude)]

ndPaths = sorted(neuralDataPaths, key=group_by_mouse_day)

session_groups = [list(g) for _, g in groupby(ndPaths, key=group_by_mouse_day)]
session_groups = sorted(session_groups)

# Constructing bins - 100 ms time bins (10Hz)

In [584]:
# Defining parameters

dtFR = 0.100 # 100 ms
fs = 1/dtFR
tmin = 0.0
tmax = 1205.0 # From catGT -maxsecs=1205.00; So always fixed for all sessions
bins = np.arange(tmin, tmax + dtFR, dtFR)

print(f"Data frequency: {fs} Hz")

Data frequency: 10.0 Hz


## Aligning histology data (Probe objects from HERBs) with the neural data

### *Herbs object explanation for Neuropixels 1.0:*

*probe["data"]["sites_loc_b"]* will output the probe coordinates along the shank relative to bregma in millimeters. 
It is array with dimensions (4, 121, 3): 

- 4    = four Neuropixels site columns
- 121  = anatomic positions along the depth of each column (herbs anatomy resolution/sampling)
- 3    = 3D coordinate for each site (ML, AP, DV). This coordinate system was checked against the real probe trajectories

site 0 (index 0 out 121) represents the deepest part of an active probe. This can be confirmed by using the anatomical regions in *probe["data"]["label_name"]*.
You will see that the deepest anatomical structure represents the first index, whereas the most superficial brain region is in the last index. You can also use
*probe["data"]["insertion_coords"] / atlasResolution* and *probe["data"]["terminus_coords"] / atlasResolution* and compare with the original *probe["data"]["sites_loc_b"]*

Note that the insertion and terminus coordinates will differ slighly from the upper and lower range of the *probe["data"]["sites_loc_b"]* array. This is because
*probe["data"]["sites_loc_b"]* represents the coordinates of the electrode coated regions of the probe, whereas insertion is the contact point of the probe with the brain, 
and the terminus coordinates are the probe's tip.



"sites_loc_b" values explanation

| Coordinate | Interpretation |
|---|---|
| Smaller ML | More lateral |
| Smaller AP | More posterior |
| Smaller DV | Deeper |


Mapping between Kilosort probe geometry and *probe["data"]["sites_loc_b"]* from HERBS

In the *channel_positions.npy* file from Kilosort4 (or *chanMap.json*) you will find channel positions: **[xc, yc]**
"xc" represents one of the 4 columns of the probe and "yc" the depth along the shank. From left to right in the probe image, the 2 electrodes at the tip of the shank have coordinates of 
[16, 0] and [48, 0], respectively. In the second row, from left to right, the coordinates are [0, 20] and [32, 0]


<img src="https://brainmapportal-live-4cc80a57cd6e400d854-f7fdcae.divio-media.net/filer_public/88/00/8800429b-8811-471f-9f04-82d19b3851b0/neuropixels_visual_coding_cms_images-04.png" width="400">

| HERBS index | Electrode position | HERBS location | HERBs coodinates example (LS probe) |
|---:|---:|---| ---|
| 1 | [16, 0] | probe["data"]["sites_loc_b"][1][0] | [  1.83108815 -56.77221097 -98.01574371] |
| 3 | [48, 0] | probe["data"]["sites_loc_b"][3][0] | [  1.83982775 -56.29229054 -98.01574371] |
| 0 | [0, 20] | probe["data"]["sites_loc_b"][0][0] | [  1.63758298 -56.92871367 -97.6640867 ] |
| 2 | [32, 20] | probe["data"]["sites_loc_b"][2][0] | [  1.64632258 -56.44879324 -97.6640867 ] |

In the herbs probe object, column indeces 1 and 3 are the deepest electrodes. These represent electrodes (16,0) and (48,0) in the probe geometry example above



In [ ]:
# Full probe geometry (384 channels) - DREDge interpolation can change this number slighly

with open(r"J:\project_trainingAggression\Code\Neurpixels_aggressionTraining\fullProbeGeometry.pkl", "rb") as prb:
    fullProbeGeometry = pickle.load(prb)
probeOrigGeom = [np.array(f) for f in zip(fullProbeGeometry["xc"],fullProbeGeometry["yc"])]

np_sites = pd.DataFrame({"np_chanMap": np.asarray(fullProbeGeometry["chanpMap"]),
                            "np_xc": np.asarray(fullProbeGeometry["xc"]),
                            "np_yc": np.asarray(fullProbeGeometry["yc"])})
    
# Assign column number based on x-position
herbsCol_top_npCol_map = {16:0,
                          48:1,
                          0:2,
                          32:3}

np_sites["np_col"] = np_sites["np_xc"].map(herbsCol_top_npCol_map)

def alignHistology2NeuralData(probePath, good_units, np_sites, it):

    print("Aligning HERBs probe tracks to Firing Rate matrix...")

    chMp = probePath.parent.joinpath("chanMap.json")

    dayKey = [s for s in str(chMp).split("\\") if "Day" in s]
    mouseFolder = str(chMp).split(dayKey[0])[0]

    probe_matches = list(Path(mouseFolder).rglob(rf"*herbsProbeTrajectory/probes/probe {it + 1}.pkl"))

    if len(probe_matches) == 0:
        print("No HERBS histology annotations")
        return None

    probeObj = probe_matches[0]

    if probeObj.exists():

        # -----------------------------
        # 1. Convert HERBS probe object sites to table
        # -----------------------------

        print("Loading...", probeObj)

        with open(probeObj, "rb") as f:
            herbsPrb = pickle.load(f)
            
        herbs = herbsPrb["data"]

        rows = []

        for col_i, (locs, vox, labels) in enumerate(zip(herbs["sites_loc_b"], herbs["sites_vox"], herbs["sites_label"])):

            for site_i, (xyz, ijk, label) in enumerate(zip(locs, vox, labels)):
                
                rows.append({
                    "herbs_col": col_i,
                    "herbs_site": site_i,
                    "ml_b": xyz[0],
                    "ap_b": xyz[1],
                    "dv_b": xyz[2],
                    "j_vox": ijk[0],
                    "i_vox": ijk[1],
                    "k_vox": ijk[2],
                    "allen_id": int(label),
                })

        herbs_sites = pd.DataFrame(rows)

        # Add Allen acronym / name from HERBS
        label_map = dict(zip(map(int, herbs["region_label"]),herbs["label_acronym"]))
        name_map = dict(zip(map(int, herbs["region_label"]),herbs["label_name"]))

        herbs_sites["region"] = herbs_sites["allen_id"].map(label_map)
        herbs_sites["region_name"] = herbs_sites["allen_id"].map(name_map)

        # -----------------------------
        # 2. Sorting HERBs probe object to match channel sorting from the sorter (Kilosort4 + cell selection) 
        # -----------------------------

        herbs_npGeometrySorted = []

        for hb_id in sorted(np.unique(herbs_sites["herbs_site"])):
            for col_id in sorted(np.unique(herbs_sites["herbs_col"])):
                herbs_npGeometrySorted.append(herbs_sites[(herbs_sites["herbs_col"] == col_id) & (herbs_sites["herbs_site"] == hb_id)])

        herbs_npGeometrySorted = pd.concat(herbs_npGeometrySorted, ignore_index=True)

        # -----------------------------
        # 3. Aligning the re-sorted HERBs probe object to full probe geometry
        # Note: DREDge, due to interpolation, eliminates some electrodes at the edges of the probe. 
        #       So we are going to map the HERBs probe object to the full geometry first
        # -----------------------------

        # Nearest-neighbor mapping
        idx = np.linspace(0, len(herbs_npGeometrySorted.index)-1, len(fullProbeGeometry["chanpMap"])).astype(int)
        herbs_resampled = herbs_npGeometrySorted.iloc[idx].reset_index()

        alignedHistology = herbs_resampled.merge(np_sites,
                                                 left_index=True,
                                                 right_index =True,
                                                 how="right").drop(columns = "index")

        # -----------------------------
        # 3. Segmenting the aligned data along the post processed probe data with DREDge
        # -----------------------------

        with open(chMp, "r") as p:
            sorterPrb = json.load(p)

        probeGeom = [np.array(f) for f in zip(sorterPrb["xc"],sorterPrb["yc"])]

        # compare each row of big with first/last row of small
        lower_idx = np.where((probeOrigGeom == probeGeom[0]).all(axis=1))[0][0]
        upper_idx = np.where((probeOrigGeom == probeGeom[-1]).all(axis=1))[0][0]

        trim_alignedHistology = alignedHistology.loc[lower_idx:upper_idx].reset_index().drop(columns = "index")

        firstE = trim_alignedHistology["np_chanMap"][0]
        trim_alignedHistology["np_chanMap"] = trim_alignedHistology["np_chanMap"] - firstE

        # -----------------------------
        # 3. Align histology and probe geometry data to cells/clusters in cluster_info.tsv
        # -----------------------------

        goodUnits_alignedHistology = trim_alignedHistology.merge(good_units,
                                                                 left_on="np_chanMap",
                                                                 right_on="ch",
                                                                 how="right")


        goodUnits_alignedHistology["np_chanMap"] = goodUnits_alignedHistology["np_chanMap"] + firstE
        goodUnits_alignedHistology = goodUnits_alignedHistology.rename(columns = {"ch": "ch_cluster_info"})

        goodUnits_alignedHistology.to_csv(f"{probeObj.parent}/alignedHistologyInfo_{str(probeObj.stem).replace(' ', '')}.csv", index=False)
        
        anatomyPrb = np.column_stack([goodUnits_alignedHistology["cluster_id"].values, goodUnits_alignedHistology["region_name"].values])
        print(good_units)
    else:
        print(" ---------No HERBS histology annotations for this session ---------")

    return anatomyPrb



# Firing rate matrix per session (x3 probes)

In [646]:
# Generates RAW Firing Rate matrix for a single multiprobe session
def generateSession_histAlign_neuralDataFR(sessionProbeDataPaths, bins, dtFR, tCutSamp, np_sites): 

    print("Generating Firing Rate matrix...")

    perSessionFR = []
    perSessionAnatomy = []
    for it, probePath in enumerate(sessionProbeDataPaths):

        sessionStr = re.search(r"\d+_m\d+_obs\d+_g\d+_imec\d+", str(probePath))
        print(f"Loading: {sessionStr.group()}")

        cluster_info = pd.read_csv(probePath.joinpath("cluster_info.tsv"), sep ="\t")
        spikeClusters = np.load(probePath.parent.joinpath("spike_clusters.npy")).astype(int)
        spikeSeconds = np.load(probePath.parent.joinpath(f"spike_seconds_imec{it}_adj.npy")).astype(float)

        cluster_info = cluster_info.sort_values("ch") # Sorted along probe geometry/channel index | Important for proper alignment with Histology data

        good_units = cluster_info.loc[cluster_info["group"] == "good", ["cluster_id", "ch"]].copy()
        goodClusters = good_units["cluster_id"].to_numpy()
        nGood = len(good_units)
        nBins = len(bins) - 1 

        perProbeFR = np.zeros(shape = (nGood, nBins))
        for cell, clu in enumerate(goodClusters):
            perCellSpikes = spikeSeconds[spikeClusters == clu]
            perCellFR, _ = np.histogram(perCellSpikes, bins=bins)
            
            perProbeFR[cell, :] = perCellFR/dtFR
        
        perSessionFR.append(perProbeFR)

        anatomyPrb = alignHistology2NeuralData(probePath, good_units, np_sites, it)
        perSessionAnatomy.append(anatomyPrb)

    perSessionFR = np.concatenate(perSessionFR, axis=0) 
    perSessionAnatomy = np.concatenate(perSessionAnatomy) 

    if perSessionFR.shape[0] != perSessionAnatomy.shape[0]:
        print(" ========================= WARNING: Something went wrong with the HERBS Histology - Neuropixels alignment =========================")

    print(f"Number of cells: {np.shape(perSessionFR)[0]}\n")
        
    return perSessionFR[:, 0:tCutSamp+1], perSessionAnatomy

In [587]:
kernelWidthSec = 1 # seconds
def generateSession_neuralDataFR_heatmap(perSessionFR, dtFR, kernelWidthSec, sessionPath, session_name, tCutSamp):

    print("Saving Gaussian filtered and Zscored Firing Rate matrix heatmap...")

    # Smoothing the data: Gaussian filter
    trunc =  3.0
    sigmaSec = kernelWidthSec / (2 * trunc) 
    sigmaBin = sigmaSec / dtFR
    smoothed_FR = gaussian_filter1d(perSessionFR, sigmaBin, axis=1, truncate = trunc)

    # Z-scoring per neuron
    mean_FR = smoothed_FR.mean(axis=1, keepdims=True)
    std_FR = smoothed_FR.std(axis=1, keepdims=True)
    std_FR[std_FR == 0] = 1e-8

    zFR = (smoothed_FR - mean_FR) / std_FR

    if(np.isnan(zFR).any()):
        print("This FR matrix has NAN values after Z scoring <---------------------------------------------------------------")

    # Saving the heatmap
    plt.figure(figsize=(12, 8))
    plt.imshow(
        zFR,
        aspect='auto',
        origin='lower',
        extent=[0, 1200, 0, zFR.shape[0]],
        cmap="jet",
        vmin= -1.0,
        vmax= 1.0
    )
    plt.colorbar(label='Z-score')
    plt.xticks(np.arange(tmin, tmax, 60), np.arange(0, 21, 1))
    plt.xlabel('Time (min)')
    plt.ylabel('Neuron')
    plt.title(f'Firing rate heatmap: {session_name} | nCells = {zFR.shape[0]}')

    out_path = sessionPath / f"{session_name}_neuralDataFR_heatmap_{int(dtFR*1000)}msBins_{kernelWidthSec}sGaussFilt.png"
    plt.savefig(out_path, 
                dpi=300, 
                bbox_inches='tight')
    plt.close()

# Peripheral Data

In [588]:
def generateSession_peripheralData(sessionPath, bins, dtFR, tCutSamp):

    peripheralData = dict()

    sessionStr = str(next(sessionPath.rglob("*nidq.bin")).name)
    print(f"{sessionStr}: Generating TLL condition's matrix...")

    with open(next(sessionPath.rglob("*nidq.xid*_2_50.txt")), "r") as milk:
        milk_ON = milk.read()

    with open(next(sessionPath.rglob("*nidq.xd*_3_0.txt")), "r") as redLightOn:
        redL_ON = redLightOn.read()

    with open(next(sessionPath.rglob("*nidq.xid*_3_360000.txt")), "r") as redLightOFF:
        redL_OFF = redLightOFF.read()
    
    with open(next(sessionPath.rglob("*nidq.xd*_4_1e+06.txt")), "r") as gate:
        gateOpen = gate.read()

    with open(next(sessionPath.rglob("*nidq.xd*_5_3500.txt")), "r") as urine:
        urine_ON = urine.read()

 
    milkTTL = np.zeros(len(bins)-1)
    urineTTL = np.zeros(len(bins)-1)
    whiteLightTTL = np.zeros(len(bins)-1)
    redLightTTL = np.zeros(len(bins)-1)
    gateTTL = np.zeros(len(bins)-1)
  
    milkOnsets = [np.argmin(np.abs(bins - float(milk))) for milk in milk_ON.split()]
    milk_ON_duration = 0.050 # seconds
    milkOffset = int(np.ceil(milk_ON_duration/dtFR))
    for m_onset in milkOnsets:
        milkTTL[m_onset:m_onset+milkOffset] = 1 # Not (m_onset+milkOffset + 1) because milk duration is indeed only one sample (50 ms) at 100ms bins

    urineOnsets = [np.argmin(np.abs(bins - float(urine))) for urine in urine_ON.split()]
    urine_ON_duration = 3.62 # Calculated from mean(*nidq.xd*_5_11000.txt - *nidq.xd*_5_3500.txt)
    urineOffset = int(urine_ON_duration/dtFR)
    for u_onset in urineOnsets:
        urineTTL[u_onset:u_onset+urineOffset + 1] = 1 

    gateOnset = np.argmin(np.abs(bins - float(gateOpen.strip())))
    gateTTL[gateOnset:] = 1

    redLightOnset = [np.argmin(np.abs(bins - float(t))) for t in redL_ON.split()]

    if redL_OFF != '':
        whiteLightOnset = np.argmin(np.abs(bins - float(redL_OFF.strip())))

        whiteLightTTL[whiteLightOnset:redLightOnset[1]] = 1
        redLightTTL = 1 - whiteLightTTL

    else:
        redLightTTL = 1 - whiteLightTTL

    peripheralData["milkTTL"] = milkTTL[0:tCutSamp+1]
    peripheralData["urineTTL"] = urineTTL[0:tCutSamp+1]
    peripheralData["redLightTTL"] = redLightTTL[0:tCutSamp+1]
    peripheralData["whiteLightTTL"] = whiteLightTTL[0:tCutSamp+1]
    peripheralData["gateTTL"] = gateTTL[0:tCutSamp+1]

    return peripheralData

# Manual Annotations 

In [589]:
# Annotation files from BENTO (MATLAB)

def isfloat(s):
    try:
        if not s.isdigit():
            float(s)
            return True
        else:
            return False
    except ValueError:
        return False

def generatingSession_behavioralData(sessionPath, bins, tCutSamp):

    behavioralData = dict()
    annotationsPath = next(sessionPath.rglob("*.annot"), None)

    if (annotationsPath != None):
        print(f"{annotationsPath.name}: Generating behavioral matrix...")
        
        with open (annotationsPath, "r") as behav:
            b = behav.read()

        for row in b.split("\n"):

            if row.startswith("Annotation framerate:"):
                fsAnnot = float(row.split(":")[1])

            if row.startswith(">"):
                behavLabel = row[1:]
                behavioralData[behavLabel] = np.zeros(len(bins)-1)[0:tCutSamp+1]

            r = row.split()
            if (len(r) == 3) and (all([p.isdigit() for p in r]) == True): # When events are saved in samples in Bento
                start, stop, duration = map(int, r)
                startT = np.argmin(np.abs(bins - start/fsAnnot))
                stopT = np.argmin(np.abs(bins - stop/fsAnnot))
            elif (len(r) == 3) and (all([isfloat(p) for p in r]) == True): # When events are saved in seconds in Bento
                start, stop, duration = map(float, r)
                startT = np.argmin(np.abs(bins - start))
                stopT = np.argmin(np.abs(bins - stop))
            else:
                continue

            behavioralData[behavLabel][startT:stopT+1] = 1

        return behavioralData
        
    else:
        print("----- No manual annotations found for this session -----")
        return None


# Main Loop

In [660]:
allCells = 0

for session_prbPaths in session_groups[0:1]:    
    session_name = [name for name in str(session_prbPaths[0]).split("\\") if name.startswith("catgt_")][0]
    sessionPath = Path(str(session_prbPaths[0]).split(f"{session_name}")[0])
    session_name = session_name[6:-3]
    
    tCut = 1200
    tCutSamp = np.argmin(np.abs(bins - tCut))

    print(f"\n========================== Generating files for session: {session_name} ==========================\n")

    perSessionFR, perSessionAnatomy = generateSession_histAlign_neuralDataFR(session_prbPaths, bins, dtFR, tCutSamp, np_sites)
    generateSession_neuralDataFR_heatmap(perSessionFR, dtFR, kernelWidthSec, sessionPath, session_name, tCutSamp)
    peripheralData = generateSession_peripheralData(sessionPath, bins, dtFR, tCutSamp)
    behavioralData = generatingSession_behavioralData(sessionPath, bins, tCutSamp)

    np.save(sessionPath / f"{session_name}_neuralDataFR_{int(dtFR*1000)}msBins",
            perSessionFR,
            allow_pickle=True)

    '''
    np.save(sessionPath / f"{session_name}_histologyAlignedData",
            perSessionAnatomy,
            allow_pickle = True)
    '''
    with open(sessionPath / f"{session_name}_peripheralData_{int(dtFR*1000)}msBins.pkl", "wb") as prData:
        pickle.dump(peripheralData, prData)
    savemat(sessionPath / f"{session_name}_peripheralData_{int(dtFR*1000)}msBins.mat", peripheralData)
    
    if behavioralData is not None:
        with open(sessionPath / f"{session_name}_behavioralData_{int(dtFR*1000)}msBins.pkl", "wb") as bhData:
            pickle.dump(behavioralData, bhData)
        savemat(sessionPath / f"{session_name}_behavioralData_{int(dtFR*1000)}msBins.mat", behavioralData)

    allCells+=np.shape(perSessionFR)[0]

print(f"\n\n=============================================== Total Number of cells: {allCells} =============================================== ")


========================== Generating files for session: 20250817_m975826_obs1 ==========================

Generating Firing Rate matrix...
Loading: 20250817_m975826_obs1_g0_imec0
Aligning HERBs probe tracks to Firing Rate matrix...
Loading... J:\project_trainingAggression\Data\20250817_mouse975826\herbsProbeTrajectory\probes\probe 1.pkl
     cluster_id   ch
44           44   55
77           77  109
129         129  221
151         151  299
157         157  303
159         159  306
160         160  307
179         179  316
183         183  316
184         184  317
181         181  319
190         190  323
197         197  330
204         204  341
217         217  356
Loading: 20250817_m975826_obs1_g0_imec1
Aligning HERBs probe tracks to Firing Rate matrix...
Loading... J:\project_trainingAggression\Data\20250817_mouse975826\herbsProbeTrajectory\probes\probe 2.pkl
     cluster_id   ch
5             5    5
7             7    8
15           15   15
32           32   32
28           28   